In [0]:

# CELL 1 — ADF Notebook Parameters

dbutils.widgets.text(
    "batch_id",
    "merge_test_batch_003"
)

dbutils.widgets.text(
    "source_table",
    "dbw_retail_dataeng.retail_part2_dev.retail_source_snapshot"
)

dbutils.widgets.text(
    "target_table",
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

dbutils.widgets.text(
    "tracking_table",
    "dbw_retail_dataeng.retail_part2_dev.retail_batch_tracking"
)

# Read parameter values
batch_id = dbutils.widgets.get("batch_id")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
tracking_table = dbutils.widgets.get("tracking_table")

print("Batch ID:", batch_id)
print("Source Table:", source_table)
print("Target Table:", target_table)
print("Tracking Table:", tracking_table)


In [0]:

# CELL 2 — Parameter Validation and Batch Status

from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Validate required parameters
parameters = {
    "batch_id": batch_id,
    "source_table": source_table,
    "target_table": target_table,
    "tracking_table": tracking_table
}

for name, value in parameters.items():
    if not value or not value.strip():
        raise ValueError(f"Missing required parameter: {name}")

# Verify the required tables exist
for table_name in [
    source_table,
    target_table,
    tracking_table
]:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Table not found: {table_name}")

# Check whether the batch is already processed
processed_count = (
    spark.table(tracking_table)
    .filter(
        (F.col("batch_id") == batch_id) &
        (F.col("processing_status") == "SUCCESS")
    )
    .count()
)

batch_already_processed = processed_count > 0

print("Parameter validation: PASS")
print("Required tables: FOUND")
print("Batch ID:", batch_id)
print(
    "Batch status:",
    "ALREADY PROCESSED"
    if batch_already_processed
    else "READY FOR PROCESSING"
)


In [0]:

# CELL 3 — Select Incremental Batch

from pyspark.sql import functions as F

# Development batch mapping
batch_mapping = {
    "merge_test_batch_001": 1,
    "merge_test_batch_002": 2,
    "merge_test_batch_003": 2,
    "merge_test_batch_004": 2
}

# Validate the requested batch ID
if batch_id not in batch_mapping:
    raise ValueError(
        f"Unknown development batch ID: {batch_id}"
    )

# Read persisted source snapshot
df_source = spark.table(source_table)

# Assign records using the same stable hash
# used in our earlier development tests
df_assigned = df_source.withColumn(
    "batch_number",
    F.when(
        F.pmod(
            F.xxhash64("test_record_id"),
            F.lit(2)
        ) == 0,
        1
    ).otherwise(2)
)

# Select requested batch
selected_batch_number = batch_mapping[batch_id]

df_selected_batch = (
    df_assigned
    .filter(
        F.col("batch_number") == selected_batch_number
    )
    .drop("batch_number")
)

print("Selected Batch ID:", batch_id)
print("Selected Batch Records:", df_selected_batch.count())
print("Already Processed:", batch_already_processed)


In [0]:

# CELL 4 — Automated Data Quality Validation

from pyspark.sql import functions as F

# Calculate data quality metrics
quality_metrics = df_selected_batch.agg(
    F.count("*").alias("total_records"),

    F.countDistinct("test_record_id").alias("unique_ids"),

    F.sum(
        F.col("test_record_id").isNull().cast("int")
    ).alias("null_ids"),

    F.sum(
        F.col("InvoiceNo").isNull().cast("int")
    ).alias("null_invoices"),

    F.sum(
        F.col("StockCode").isNull().cast("int")
    ).alias("null_stock_codes"),

    F.sum(
        F.col("Quantity").cast("int").isNull().cast("int")
    ).alias("invalid_quantities"),

    F.sum(
        F.col("UnitPrice").cast("double").isNull().cast("int")
    ).alias("invalid_prices")
).first()

# Define validation rules
quality_rules = {
    "Batch is not empty":
        quality_metrics["total_records"] > 0,

    "Record IDs are unique":
        quality_metrics["total_records"] ==
        quality_metrics["unique_ids"],

    "Record IDs are not null":
        quality_metrics["null_ids"] == 0,

    "Invoice numbers are present":
        quality_metrics["null_invoices"] == 0,

    "Stock codes are present":
        quality_metrics["null_stock_codes"] == 0,

    "Quantities are numeric":
        quality_metrics["invalid_quantities"] == 0,

    "Unit prices are numeric":
        quality_metrics["invalid_prices"] == 0
}

# Display validation results
for rule, passed in quality_rules.items():
    print(f"{rule}: {'PASS' if passed else 'FAIL'}")

batch_valid = all(quality_rules.values())

print(
    "\nOVERALL DATA QUALITY:",
    "PASS" if batch_valid else "FAIL"
)

if not batch_valid:
    raise ValueError(
        f"Data quality validation failed for {batch_id}"
    )


In [0]:

# CELL 5 — Delta MERGE with Batch Skipping

from delta.tables import DeltaTable

# Recheck the tracking table at execution time
already_processed = (
    spark.table(tracking_table)
    .filter(
        (F.col("batch_id") == batch_id) &
        (F.col("processing_status") == "SUCCESS")
    )
    .limit(1)
    .count() > 0
)

if already_processed:
    print(f"SKIPPED: {batch_id} already processed.")
    processing_result = "SKIPPED"

else:
    if not batch_valid:
        raise ValueError(
            f"MERGE blocked: Data quality failed for {batch_id}"
        )

    print("Data quality: PASS")
    print("Starting Delta Lake MERGE...")

    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            df_selected_batch.alias("source"),
            "target.test_record_id = source.test_record_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    processing_result = "MERGED_NOT_TRACKED"
    print("MERGE completed successfully.")

print("Processing result:", processing_result)
print("Target records:", spark.table(target_table).count())


In [0]:

# CELL 6 — Automatic Batch Registration

from pyspark.sql import functions as F

if processing_result == "SKIPPED":

    print(f"Batch {batch_id} was already processed.")
    print("No new tracking record required.")

elif processing_result == "MERGED_NOT_TRACKED":

    # Count records in the incoming batch
    record_count = df_selected_batch.count()

    # Create metadata tracking record
    tracking_record = (
        spark.createDataFrame(
            [(batch_id, source_table, record_count)],
            ["batch_id", "source_file", "records_processed"]
        )
        .withColumn(
            "file_fingerprint",
            F.lit(None).cast("string")
        )
        .withColumn(
            "processing_status",
            F.lit("SUCCESS")
        )
        .withColumn(
            "processed_at",
            F.current_timestamp()
        )
        .select(
            "batch_id",
            "source_file",
            "file_fingerprint",
            "records_processed",
            "processing_status",
            "processed_at"
        )
    )

    # Append successful batch metadata
    (
        tracking_record.write
        .format("delta")
        .mode("append")
        .saveAsTable(tracking_table)
    )

    processing_result = "SUCCESS"

    print(f"SUCCESS: Batch {batch_id} registered.")

else:
    raise ValueError(
        f"Unexpected processing result: {processing_result}"
    )

# Final execution summary
print("\n===== PIPELINE EXECUTION SUMMARY =====")
print("Batch ID:", batch_id)
print("Processing Status:", processing_result)
print("Target Table:", target_table)
print("Target Records:", spark.table(target_table).count())


In [0]:
%sql

SELECT current_user() AS executing_user,
       is_account_group_member('account users') AS account_user_member;


In [0]:
from pyspark.sql import functions as F

source_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_source_snapshot"
)

silver_table = (
    "dbw_retail_dataeng.retail_part2_dev.retail_merge_test"
)

df_batch004 = (
    spark.table(source_table)
    .filter(
        F.pmod(
            F.xxhash64("test_record_id"),
            F.lit(2)
        ) == 1
    )
)

total_records = df_batch004.count()
unique_records = df_batch004.select(
    "test_record_id"
).distinct().count()

test_records = df_batch004.filter(
    F.col("test_record_id") == "adf_e2e_test_001"
).count()


print("==BATCH 004 VALIDATION===")
print("Selected records:", total_records)
print("Unique record IDs:", unique_records)
print("New test record occurrences:", test_records)
print("Test record already in Silver:", already_in_silver)

assert total_records == unique_records, "Duplicate source IDs detected"
assert test_records == 1, "Test record must appear exactly once"
assert already_in_silver == 0, "Test record already exists in Silver"

print("PASS: Batch 004 is ready for ADF execution.")

